# Advising Quality Evaluation
Scores responses on Relevance, Correctness, Personalization, Non-hallucination, and Policy Consistency.

In [ ]:
import requests, uuid, json, statistics
import pandas as pd
import matplotlib.pyplot as plt

API_BASE = 'http://localhost:8000'

# Scoring rubric (1-5 scale):
# Relevance:        5=directly answers, 4=mostly relevant, 3=partially, 2=tangential, 1=irrelevant
# Correctness:      5=fully accurate, 4=minor errors, 3=partially correct, 2=mostly wrong, 1=wrong
# Personalization:  5=uses student context well, 3=generic but ok, 1=ignores context
# Non-hallucination:5=grounded, 3=minor unverifiable claims, 1=fabricated facts
# Policy Consistency:5=aligns with university policy, 3=neutral, 1=contradicts policy

# These are auto-scored using keyword and heuristic checks (in a real eval, use human annotators or LLM judge)

advising_scenarios = [
    {
        'id': 'S1',
        'context': {'program': 'BTech', 'year': 2, 'department': 'Computer Science'},
        'query': 'Can I take ML301 as an elective this semester?',
        'must_contain': ['prerequisite', 'AI101', 'MATH301', 'STAT201'],
        'must_not_contain': ['you can take it without prerequisites'],
        'policy_keywords': ['prerequisite', 'eligible'],
    },
    {
        'id': 'S2',
        'context': {'program': 'MBA', 'year': 1, 'nationality': 'international'},
        'query': 'What documents do I need for visa renewal and FRRO registration?',
        'must_contain': ['FRRO', 'visa', 'international', 'ISC', '14 days'],
        'must_not_contain': ['not applicable'],
        'policy_keywords': ['FRRO', 'registration', '14 days'],
    },
    {
        'id': 'S3',
        'context': {'program': 'BTech', 'year': 3, 'cgpa': 8.9},
        'query': 'Am I eligible for the merit scholarship?',
        'must_contain': ['merit', 'scholarship', '8.5', 'CGPA'],
        'must_not_contain': ['not eligible'],
        'policy_keywords': ['CGPA', '8.5', 'scholarship'],
    },
    {
        'id': 'S4',
        'context': {'program': 'BTech', 'year': 4},
        'query': 'My attendance is 70% in CS401, can I still sit in the exam?',
        'must_contain': ['75%', 'debarred', 'condonation', 'HOD'],
        'must_not_contain': ['yes, 70% is fine'],
        'policy_keywords': ['75%', 'debarred', 'attendance'],
    },
    {
        'id': 'S5',
        'context': {'program': 'PhD', 'year': 1},
        'query': 'What stipend will I get and how is tuition handled?',
        'must_contain': ['31,000', 'fellowship', 'JRF', 'tuition'],
        'must_not_contain': [],
        'policy_keywords': ['stipend', 'fellowship', 'tuition'],
    },
]

print(f'{len(advising_scenarios)} advising scenarios loaded.')

In [ ]:
def setup_session(session_id, context):
    requests.post(f'{API_BASE}/session/{session_id}/update', json=context, timeout=10)

def get_response(session_id, query):
    try:
        r = requests.post(f'{API_BASE}/chat',
            json={'session_id': session_id, 'message': query}, timeout=90)
        data = r.json()
        return data.get('response', ''), data.get('tool_used', ''), data.get('sources', [])
    except:
        return '', '', []

def score_response(response, scenario):
    resp_lower = response.lower()
    
    # Relevance: contains must_contain keywords
    kw_hits = sum(1 for kw in scenario['must_contain'] if kw.lower() in resp_lower)
    relevance = min(5, 1 + int(kw_hits / max(len(scenario['must_contain']), 1) * 4))
    
    # Correctness: same
    correctness = relevance  # simplified; in practice, use human or LLM judge
    
    # Personalization: mentions program/year context
    ctx = scenario['context']
    prog_hit = ctx.get('program','').lower() in resp_lower
    nat_hit = 'international' in resp_lower if ctx.get('nationality') == 'international' else True
    personalization = 5 if (prog_hit and nat_hit) else 3 if prog_hit else 2
    
    # Non-hallucination: does NOT contain bad phrases
    bad_hits = sum(1 for phrase in scenario['must_not_contain'] if phrase.lower() in resp_lower)
    non_hallucination = 5 if bad_hits == 0 else max(1, 5 - bad_hits * 2)
    
    # Policy consistency: contains policy keywords
    pol_hits = sum(1 for kw in scenario['policy_keywords'] if kw.lower() in resp_lower)
    policy_consistency = min(5, 1 + int(pol_hits / max(len(scenario['policy_keywords']), 1) * 4))
    
    overall = round(statistics.mean([relevance, correctness, personalization, non_hallucination, policy_consistency]), 1)
    return {
        'Relevance': relevance,
        'Correctness': correctness,
        'Personalization': personalization,
        'Non-Hallucination': non_hallucination,
        'Policy Consistency': policy_consistency,
        'Overall': overall,
    }

eval_results = []
print('Running advising quality evaluation...')
for scenario in advising_scenarios:
    session_id = f'eval_{scenario["id"]}_{uuid.uuid4().hex[:6]}'
    setup_session(session_id, scenario['context'])
    response, tool_used, sources = get_response(session_id, scenario['query'])
    scores = score_response(response, scenario)
    eval_results.append({
        'Scenario': scenario['id'],
        'Query': scenario['query'][:50] + '...',
        'Tool Used': tool_used,
        **scores,
    })
    print(f'  {scenario["id"]}: Overall={scores["Overall"]}/5 | Tool={tool_used}')

df = pd.DataFrame(eval_results)
print('\n', df[['Scenario','Relevance','Correctness','Personalization','Non-Hallucination','Policy Consistency','Overall']].to_string(index=False))

In [ ]:
metric_cols = ['Relevance', 'Correctness', 'Personalization', 'Non-Hallucination', 'Policy Consistency']
means = df[metric_cols].mean()
print('Average Scores across all scenarios:')
for m, v in means.items():
    stars = '★' * int(v) + '☆' * (5 - int(v))
    print(f'  {m:<22}: {v:.2f}/5  {stars}')
print(f'\n  Overall Average: {df["Overall"].mean():.2f}/5')

In [ ]:
import numpy as np

angles = np.linspace(0, 2*np.pi, len(metric_cols), endpoint=False)
values = means.values.tolist()
values += values[:1]  # close
angles = np.concatenate([angles, [angles[0]]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Radar chart
ax1 = plt.subplot(121, polar=True)
ax1.plot(angles, values, 'o-', color='steelblue', linewidth=2)
ax1.fill(angles, values, alpha=0.25, color='steelblue')
ax1.set_xticks(angles[:-1])
ax1.set_xticklabels([m.replace(' ', '\n') for m in metric_cols], fontsize=9)
ax1.set_ylim(0, 5)
ax1.set_title('Average Advising Quality\n(Radar Chart)', pad=20)

# Bar chart per scenario
ax2 = plt.subplot(122)
x = np.arange(len(advising_scenarios))
ax2.bar(x, df['Overall'], color='steelblue', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(df['Scenario'])
ax2.set_ylim(0, 5.5)
ax2.axhline(4.0, color='green', linestyle='--', label='Target (4.0)')
ax2.set_ylabel('Overall Score /5')
ax2.set_title('Overall Advising Quality by Scenario')
ax2.legend()

plt.tight_layout()
plt.savefig('advising_quality.png', dpi=150)
plt.show()